# Tilslutning til hardware

**RESUMÉ**: *I denne øvelse lærer du, hvordan du opsætter og tilslutter din ChipWhisperer-hardware. Vi gennemgår også, hvordan du kompilerer firmware til din target-mikrokontroller, hvordan du optager strømmålinger, og hvordan du kommunikerer med target-enheder.*

*Alle de API-kald, vi bruger her, er dokumenteret på [ChipWhisperer readthedocs](https://chipwhisperer.readthedocs.io/en/latest/) siden, så du er velkommen til at åbne den i en anden fane og følge med der.*

**LÆRINGSMÅL:**
* Opsætning af din ChipWhisperer-hardware
* Brug af ChipWhisperer Python API til at tilslutte til din hardware
* Kommunikation med target
* Optagelse af en strømmåling

## Forudsætninger

Vent - inden du fortsætter, skal du sikre dig, at du har gjort følgende:

* ☑ Valideret din opsætning ved hjælp af [ChipWhisperer Opsætningstest](./ChipWhisperer%20Opsaetningstest.ipynb) notebooken.
* ☑ Gennemgået Jupyter-introduktionen.

## Fysisk opsætning

### CW-Husky med CW313/SAM4S

De eneste forbindelser, du skal oprette, er med 20-pin stikket og de to SMA-kabler (Crowbar til J3 og Pos til J4). De eneste indstillinger, du skal bekræfte, er, at JP2 er sat til T-GPIO4, og at der sidder jumper på JP1 og JP4:

<img src="img/cwhusky.jpg" alt="CW-Husky Plugged In" width=500>

I øvelserne bør du bruge `PLATFORM='CWHUSKY'`

## Tilslutning til ChipWhisperer

Nu hvor din hardware er opsat, kan vi lære, hvordan man tilslutter til den. Vi kan tilslutte til ChipWhisperer med:

In [ ]:
import chipwhisperer as cw
scope = cw.scope()

Som standard vil ChipWhisperer forsøge at autodetektere typen af enhed, du kører (CWLite/CW1200, CWNano eller CWHUSKY). Se API-dokumentationen for manuel angivelse af scope-typen. Hvis du har flere ChipWhisperer-enheder tilsluttet, skal du angive serienummeret på den enhed, du vil tilslutte til:

```python
scope = cw.scope(sn='<some long string of numbers>')
```

For mere information, se API-afsnittet på readthedocs.

Tilslutning til target-enheden er lige så simpelt:

In [ ]:
target = cw.target(scope, cw.targets.SimpleSerial) #cw.targets.SimpleSerial can be omitted

Vi vil kun diskutere standard target-typen, som er SimpleSerial. Andre mål, som CW305, vil blive dækket i hardwarespecifikke demoer.

Nogle fornuftige standardindstillinger kan angives med:

In [ ]:
scope.default_setup()

som fra sin [dokumentation](https://chipwhisperer.readthedocs.io/en/latest/scope-api.html#chipwhisperer.scopes.OpenADC.default_setup) gør følgende for CWHusky:

* Indstiller scope-forstærkningen til 25dB
* Indstiller scope til at optage 5000 samples
* Indstiller scope-offset til 0 (dvs. det begynder at optage, så snart det udløses)
* Indstiller scope-triggeren til stigende flanke
* Sender et 7,37MHz clock til målet på HS2
* Clocker scope ADC'en ved 4\*7,37MHz. Bemærk, at dette er *synkront* med target-clock på HS2
* Tildeler GPIO1 som seriel RX
* Tildeler GPIO2 som seriel TX

Og det er det! Din ChipWhisperer er nu opsat og klar til at angribe et mål.

**BEMÆRK: Du skal frakoble scope/target, inden du tilslutter igen, som du ville gøre i en anden notebook. Dette kan gøres med `scope.dis()` og `target.dis()`**

## Kompilering og upload af firmware

Det næste trin i angrebet på et target er at få noget firmware kompileret og uploadet til det. Det meste firmware til de fleste ChipWhisperer-target kan bygges ved hjælp af vores kompileringssystem, forudsat at du har den korrekte kompiler installeret (se https://chipwhisperer.readthedocs.io/en/latest/windows-install.html for info om kompilere).

Firmware skal bygges på kommandolinjen. Heldigvis kan vi, takket være Jupyter, køre en kommando i en notebook som følger:

In [ ]:
%%bash
cd ../firmware/mcu/simpleserial-base/
make PLATFORM=CWHUSKY CRYPTO_TARGET=NONE

Du bør se en lang liste over `PLATFORM`'er at bygge til. Vi lod `PLATFORM` stå tom i kommandoen ovenfor, så byggesystemet i stedet udskrev en liste over understøttede platforme. Udfyld din platform, kør byggekommandoen igen, og din firmware bør være succesfuldt bygget.

Der er to mulige måder at uploade firmware til dit mål på:

1. ChipWhisperer har indbygget support til XMEGA, STM32F*, AVR og SAM4S bootloadere. Disse kan bruges som følger:

In [ ]:
#cw.program_target(scope, cw.programmers.XMEGAProgrammer, "path/to/firmware.hex")
#cw.program_target(scope, cw.programmers.STM32FProgrammer, "path/to/firmware.hex")
#cw.program_target(scope, cw.programmers.AVRProgrammer, "path/to/firmware.hex")
cw.program_target(scope, cw.programmers.SAM4SProgrammer, "../firmware/mcu/simpleserial-base/simpleserial-base-CWHUSKY.hex")

2. For andre mål skal du bruge en ekstern programmer eller en debugger til at flashe firmware til target.

Uanset hvad din situation er, upload den firmware, vi byggede tidligere, til target-enheden. Dernæst lærer vi, hvordan man optager strømmålinger og kommunikerer med target.

## Kommunikation med målet

Kommunikation med target, som sker gennem `SimpleSerial target`-objektet, vi fik tidligere, er opdelt i to kategorier:

1. Rå seriel via `target.read()`, `target.write()`, `target.flush()` osv.

1. SimpleSerial-kommandoer via `target.simpleserial_read()`, `target.simpleserial_write()`, `target.simpleserial_wait_ack()` osv.

Den firmware, vi uploadede, bruger simpleserial-protokollen (https://chipwhisperer.readthedocs.io/en/latest/simpleserial.html), så vi starter med simpleserial. Senere bruger vi de rå serielle kommandoer til at sende de samme beskeder.

Hvis du læser simpleserial-base firmwaren (`simpleserial-base.c`), vil du finde, at for simpleserial `'p'`-kommandoen vil target ekkoere kommandoen tilbage. Lad os prøve det nu:

In [ ]:
msg = bytearray([0]*16) # simpleserial uses bytearrays
target.simpleserial_write('p', msg)

Lad os tjekke, om vi fik et svar:

In [ ]:
print(target.simpleserial_read('r', 16))

Den har også en `'k'`-kommando. Prøv at sende den nu:

In [ ]:
msg = bytearray([0]*16) # simpleserial uses bytearrays
target.simpleserial_write('k', msg)

Denne kommando returnerer ikke noget til os, men den bør ACK og give os en fejlkode:

In [ ]:
print(target.simpleserial_wait_ack()) # should return 0

Simpleserial-beskeder tager generelt formen:

```python
command_character + ascii_encoded_bytes + '\n'
```

For vores første kommando er `command_character='p'` og `ascii_encoded_bytes="00"*32` (husk, at dette ikke er et binært `0x00`, men ASCII `"00"`, som har en binær værdi på `0x3030`). Prøv at sende `'p'`-kommandoen fra tidligere igen ved hjælp af `target.write()`:

In [ ]:
target.write('p' + ) # fill in the rest here

En simpel `target.read()` returnerer alle de tegn, der er sendt tilbage fra målet indtil videre. Lad os se, hvad enheden returnerede til os:

In [ ]:
recv_msg = ""

In [ ]:
recv_msg += target.read() # you might have to run this block a few times to get the full message
print(recv_msg)

SimpleSerial-kommandoerne er normalt tilstrækkelige til at kommunikere med simpleserial-firmware, men du vil have brug for de rå serielle kommandoer til nogle af de andre øvelser.

## SimpleSerial 2

Fra og med ChipWhisperer 5.4 er en ny target-kommunikationsprotokol (SimpleSerial 2) tilgængelig. Den har en række fordele i forhold til den gamle SimpleSerial-protokol:

1. Data er ukodede binære data (undtagen byte stuffing) i stedet for ASCII-kodet, hvilket betyder, at der kan sendes langt mere data pr. pakke (knap det dobbelte)
1. Den har et kommando- og underkommandofelt, der sendes til targetcallbackfunktionerne, hvilket betyder, at der kan gøres mere pr. pakke. For eksempel er simpleserial-aes firmwaren blevet ændret til at tillade, at klartekst, nøgle og/eller maske kan angives med en enkelt pakke.
1. Den har et længdefelt, hvilket betyder, at de samme målkommandoer kan tage pakker af forskellig længde afhængigt af situationen. Dette giver f.eks. mulighed for at sende både klartekst og nøgle (eller kun én) med samme kommando i simpleserial-aes.
1. Den har en 8-bit CRC (0xA6) for dataintegritet.
1. Rammer er konsistent overhead byte stuffed (COBS). Dette gør det nemt at nulstille kommunikation (gjort simpelthen ved at sende to 0x00 bytes) og hjælper med at opdage fejlformede længdebytes.

Det er dog en mere kompleks protokol. Du kan bruge den ved at kompilere firmware med `SS_VER=SS_VER_2_0` og bruge `cw.targets.SimpleSerial2`. I `sca101` og `fault101` kan den bruges ved at angive `SS_VER='SS_VER_2_0'`, når du angiver `PLATFORM` og `SCOPETYPE`.

## Optagelse af målinger

Nu hvor target er programmeret, og vi ved, hvordan vi kommunikerer med det, lad os begynde at optage strømmålinger! For at optage en måling:

1. Armér ChipWhisperer med `scope.arm()`. Den begynder at optage, så snart den udløses (som i vores tilfælde er en stigende flanke på `GPIO4`).
1. `scope.capture()` læser den optagede strømmåling tilbage og blokerer, indtil ChipWhisperer er færdig med at optage, eller scope'et timer ud. Bemærk, at fejlreturneringen fortæller dig, om scope'et timede ud eller ej. Den returnerer ikke de optagede scope-data.
1. Du kan læse de optagede strømmålinger tilbage med `scope.get_last_trace()`.

`simpleserial_base` vil udløse ChipWhisperer, når vi sender `'p'`-kommandoen. Prøv at optage et spor nu:

In [ ]:
scope.arm()
target.simpleserial_write('p', msg)
## fill in the rest...

ChipWhisperer har også en praktisk `capture_trace()`-funktion, der:

1. Valgfrit sender `'k'`-kommandoen
1. Armerer scope'et
1. Sender `'p'`-kommandoen
1. Optager sporet
1. Læser `'r'`-returmeddelelsen
1. Returnerer en `Trace`-klasse, der grupperer måledata, `'p'`-meddelelsen, `'r'`-meddelelsen og `'k'`-meddelelsen

Det er ikke altid den bedste mulighed at bruge, men det er normalt tilstrækkeligt for de fleste simpleserial-applikationer.

## Konklusion

Og det er det! Du bør nu være klar til at fortsætte til [2 - Introduktion til Clock Glitching](./2%20-%20Introduktion%20til%20Clock%20Glitching.ipynb) (eller et af vores andre kurser).

Som et afsluttende trin bør vi frakoble fra hardwaren, så den ikke forbliver "i brug" af denne notebook.

In [ ]:
scope.dis()
target.dis()